# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MahboobAli1/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [13]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.


In [14]:
import pandas as pd

df = pd.read_csv(
    "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
)

print("Shape:", df.shape)
print("\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")



Shape: (30000, 44)

Columns:
1. content_id
2. client_id
3. search_volume
4. competition
5. competition_level
6. cpc
7. content_type
8. main_intent
9. word_count
10. char_count
11. provider_used
12. model_used
13. impressions_90d
14. clicks_90d
15. pageviews_90d
16. sessions_90d
17. users_90d
18. engaged_sessions_90d
19. ai_sessions_90d
20. scroll_events_90d
21. days_with_impressions
22. days_with_sessions
23. impressions_last_30d
24. clicks_last_30d
25. sessions_last_30d
26. impressions_prev_30d
27. clicks_prev_30d
28. sessions_prev_30d
29. content_age_days
30. age_tier
31. age_tier_order
32. days_since_last_update
33. freshness_tier
34. word_count_tier
35. char_count_tier
36. ctr
37. avg_position
38. engagement_rate
39. scroll_rate
40. ai_traffic_pct
41. impression_tier
42. position_tier
43. trend_direction
44. trend_pct


### Feature vector

I use features that describe the content, search demand, historical traffic, content age, freshness, and observed engagement before the prediction point.

I exclude identifiers and fields that are derived from the target or may contain future information.

For numeric features, missing values are filled with the median calculated from the training data. For categorical features, missing values are filled with `"Unknown"` and then one-hot encoded.

The feature vector is intended for decision-support and refresh-opportunity scoring, not as a claim of causal impact.


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Target for this modeling lane
target = "trend_pct"

# Columns deliberately excluded from the feature vector
excluded_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct"
]

# Candidate features
feature_columns = [
    col for col in df.columns
    if col not in excluded_columns
]

X = df[feature_columns].copy()
y = df[target].copy()

# Identify numeric and categorical columns
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Number of features:", len(feature_columns))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nNumeric columns:")
print(numeric_features)

print("\nCategorical columns:")
print(categorical_features)

# Preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Fit preprocessing only on training data
X_train_vector = preprocessor.fit_transform(X_train)
X_test_vector = preprocessor.transform(X_test)

print("\nOriginal training shape:", X_train.shape)
print("Feature-vector training shape:", X_train_vector.shape)
print("Feature-vector test shape:", X_test_vector.shape)

Number of features: 40
Numeric features: 29
Categorical features: 11

Numeric columns:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Categorical columns:
['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']

Original training shape: (24000, 40)
Feature-vector training shape: (24000, 71)
Feature-vector test shape: (6000, 71)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

The selected features represent information that can reasonably be observed before making the refresh-opportunity prediction.

**Search-demand features**

* `search_volume`: observed search demand for the content topic.
* `competition`: observed competition measure.
* `competition_level`: categorical competition level.
* `cpc`: observed cost-per-click measure.

**Content features**

* `content_type`: type of content.
* `main_intent`: main search intent.
* `word_count`: content length measured in words.
* `char_count`: content length measured in characters.
* `word_count_tier`: categorical word-count group.
* `char_count_tier`: categorical character-count group.

**Historical traffic features**

* `impressions_90d`: impressions observed during the previous 90 days.
* `clicks_90d`: clicks observed during the previous 90 days.
* `pageviews_90d`: pageviews observed during the previous 90 days.
* `sessions_90d`: sessions observed during the previous 90 days.
* `users_90d`: users observed during the previous 90 days.
* `engaged_sessions_90d`: engaged sessions observed during the previous 90 days.
* `ai_sessions_90d`: AI-attributed sessions observed during the previous 90 days.
* `scroll_events_90d`: scroll events observed during the previous 90 days.
* `days_with_impressions`: number of observed days with impressions.
* `days_with_sessions`: number of observed days with sessions.

**Recent versus previous-period features**

* `impressions_last_30d`: impressions in the most recent observed 30-day period.
* `clicks_last_30d`: clicks in the most recent observed 30-day period.
* `sessions_last_30d`: sessions in the most recent observed 30-day period.
* `impressions_prev_30d`: impressions in the preceding observed 30-day period.
* `clicks_prev_30d`: clicks in the preceding observed 30-day period.
* `sessions_prev_30d`: sessions in the preceding observed 30-day period.

**Age and freshness**

* `content_age_days`: observed content age.
* `age_tier`: categorical content-age group.
* `age_tier_order`: ordered representation of content age.
* `days_since_last_update`: observed days since the last update.
* `freshness_tier`: categorical freshness group.

**Observed performance features**

* `ctr`: observed click-through rate.
* `avg_position`: observed average search position.
* `engagement_rate`: observed engagement rate.
* `scroll_rate`: observed scroll rate.
* `ai_traffic_pct`: observed percentage of AI-attributed traffic.
* `impression_tier`: categorical impression group.
* `position_tier`: categorical position group.

Missing numeric values are handled with the median from the training data. Missing categorical values are handled with the most frequent category. Unknown categories in the test data are ignored by the one-hot encoder.

The features are intended to represent information available before the prediction point. I excluded fields that identify records or are directly related to the target.


In [16]:
# Missing-value audit for selected features

missing_summary = (
    X.isna()
     .sum()
     .sort_values(ascending=False)
)

print("Missing values in selected features:")
print(missing_summary[missing_summary > 0])

print("\nTotal selected features:", X.shape[1])
print(
    "Features with missing values:",
    (missing_summary > 0).sum()
)

Missing values in selected features:
provider_used        21438
word_count            7699
word_count_tier       7699
char_count            7699
char_count_tier       7699
model_used            5733
competition_level     2610
search_volume         2468
competition           2468
cpc                   2468
main_intent           2374
scroll_rate            125
dtype: int64

Total selected features: 40
Features with missing values: 12


In [17]:
print("Categorical features:")
for col in categorical_features:
    print(f"- {col}: {df[col].nunique(dropna=True)} observed categories")

Categorical features:
- competition_level: 3 observed categories
- content_type: 3 observed categories
- main_intent: 4 observed categories
- provider_used: 2 observed categories
- model_used: 5 observed categories
- age_tier: 4 observed categories
- freshness_tier: 4 observed categories
- word_count_tier: 4 observed categories
- char_count_tier: 4 observed categories
- impression_tier: 4 observed categories
- position_tier: 5 observed categories


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage hunt

I treated leakage as information that would not be legitimately available at the time the prediction is made.

The main risks are:

1. Target-derived fields.
2. Future trend information.
3. Fields that may represent the outcome rather than the information available before the outcome.
4. Identifier or client-specific fields that do not describe the content opportunity.

`trend_pct` is the prediction target and is therefore excluded from the feature vector.

`trend_direction` is also excluded because it is directly related to the observed trend outcome.

I also checked the selected feature names for obvious target, future, or trend-related fields rather than assuming that all 44 columns are safe.

The preprocessing pipeline is fitted only on the training split, which prevents test-set information from being used when learning imputation or categorical encoding.


In [18]:
# Leakage / suspicious-column name audit

suspicious_terms = [
    "target",
    "label",
    "trend",
    "future",
    "outcome",
    "next",
    "post"
]

suspicious_columns = []

for col in df.columns:
    col_lower = col.lower()

    if any(term in col_lower for term in suspicious_terms):
        suspicious_columns.append(col)

print("Columns flagged by name:")
print(suspicious_columns)

print("\nExcluded columns:")
print(excluded_columns)

print("\nTarget:")
print(target)

print("\nTarget present in X:", target in X.columns)
print(
    "trend_direction present in X:",
    "trend_direction" in X.columns
)

Columns flagged by name:
['trend_direction', 'trend_pct']

Excluded columns:
['content_id', 'client_id', 'trend_direction', 'trend_pct']

Target:
trend_pct

Target present in X: False
trend_direction present in X: False


In [19]:
# Check whether excluded leakage fields accidentally entered the feature matrix

for col in excluded_columns:
    print(f"{col}: {'PRESENT' if col in X.columns else 'excluded'}")

content_id: excluded
client_id: excluded
trend_direction: excluded
trend_pct: excluded


In [20]:
# Check for exact duplicate columns between X and target

print("Target column:", target)
print("Target included in feature columns:", target in feature_columns)

assert target not in feature_columns
assert "trend_direction" not in feature_columns

print("\nLeakage assertions passed.")

Target column: trend_pct
Target included in feature columns: False

Leakage assertions passed.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields

* `content_id` — excluded because it is an identifier, not a predictive content characteristic.
* `client_id` — excluded because it identifies the client/group and is not a content feature; it could also create unwanted client-specific memorization.
* `trend_direction` — excluded because it is directly derived from the observed trend outcome and could leak target information.
* `trend_pct` — excluded from the feature vector because it is the prediction target.

I also did not add any new future-looking fields. The feature vector uses the historical and current-observation fields supplied in the dataset, while avoiding the target-derived trend fields.

The exclusions are conservative because the goal is a defensible decision-support feature vector rather than maximizing the number of columns.


In [21]:
# Final exclusion check

print("Excluded fields and reasons:\n")

exclusion_reasons = {
    "content_id": "Identifier; not a predictive content characteristic.",
    "client_id": "Client identifier; excluded to avoid client-specific memorization.",
    "trend_direction": "Target-derived trend information; leakage risk.",
    "trend_pct": "Prediction target; cannot be used as a feature."
}

for col, reason in exclusion_reasons.items():
    print(f"{col}: {reason}")

print("\nFinal feature count:", len(feature_columns))
print("Excluded count:", len(excluded_columns))

Excluded fields and reasons:

content_id: Identifier; not a predictive content characteristic.
client_id: Client identifier; excluded to avoid client-specific memorization.
trend_direction: Target-derived trend information; leakage risk.
trend_pct: Prediction target; cannot be used as a feature.

Final feature count: 40
Excluded count: 4


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.